<a href="https://colab.research.google.com/github/programminghistorian/ph-submissions/blob/gh-pages/assets/enablar-lesson-4/enablar-lesson-4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color:#800020;">'Terms and Conditions Apply': Intrepreting AI Tool Licences in Research Contexts through Natural Language Processing</h1>

This Jupyter Notebook, hosted online, offers an alternative to running a local Jupyter host, and downloading packages locally via Conda (as our main lesson instructs).

If you are using an institutional device, or have limited disk memory, this online version may prove more intuitive.

In [ ]:
# if you have not already downloaded the following packages locally

%pip install pdfplumber
%pip install pandas
%pip install spacy

<h2> Step 1: Python packages upload </h2>

In [ ]:
# import packages from pre-loaded libraries

from pathlib import Path
import pandas as pd
import pdfplumber
import warnings

<h2> Step 2: Dataset and spaCy Deployment </h2>

In [ ]:
ROOT = Path("AI_Products_TOS")  # in our case, this contains a folder structure of 29 licences,
# for your own use simply change the filepath to your own

# Sense check that the file download is correct

next(ROOT.rglob("*.pdf"))  # first PDF it finds
print("Testing:", pdf_path)

In [ ]:
# verify successful PDF extraction
text = extract_pdf_text(pdf_path)
print("Chars:", len(text))
print(text[:1500])  # preview first 1500 characters
warnings.filterwarnings("ignore", message="Could not get FontBBox from font descriptor.*") # silence warnings, after verifying that pdfplumber is accurate

In [ ]:
# optional, you can verify that pathlib has found your files, but returning the amount of files found, [1] if only working on a single licence

pdfs = list(ROOT.rglob("*.pdf"))
print("Found PDFs:", len(pdfs))
for p in pdfs[:10]:
    print(p)

In [ ]:
# import NLP parsers
import spacy
from spacy.matcher import PhraseMatcher, Matcher
nlp = spacy.load("en_core_web_sm") # main English model

<h2>Step 3: Dictionary set up</h2>

In [ ]:
# empty template, look at main lesson text for further explanation

name_of_dictionary = {

"risk or AI_theme": {

"associated phrases": [
    "phrase",
    "phrase",
    "phrase",
    ],

"patterns": [
    [
        {"term"},
    ],
    ]
},
}

In [ ]:
# dictionary phrae and pattern example, look at main lesson text for further explanation

name_of_dictionary = {

"AI_TRAINING": {

"phrases": [
    "train",
    "improve",
    "enhance the capabilities",
    "develops",
    "automated techniques",
    "generate outputs",
    "computational analysis",
],

"patterns": [

    [
        {"LEMMA": "train"}, # lemmatises word to stem, accounting for ‘train’ and ‘training’
        {"OP": "?"}, # denotes a placeholder
        {"OP": "?"},
        {"LOWER": {"IN": ["model","system","algorithm"]}} # more targeted to avoid duplication, also reduces flattening dataset through full lowercasing
    ],

    [
        {"LEMMA": "improve"},
        {"OP": "?"},
        {"OP": "?"},
        {"LOWER": {"IN": ["model","system","service"]}}
    ],

    [   {"LEMMA": "enhance"},
        {"OP": "?"},
        {"OP": "?"},
        {"LOWER": {"IN": ["model","system","algorithm", "service"]}}
    ],

    [   {"LEMMA": "develop"},
        {"OP": "?"},
        {"OP": "?"},
        {"LOWER": {"IN": ["model","system","algorithm", "service"]}}
    ],
    [   {"LEMMA": "automate"},
        {"OP": "?"},
        {"OP": "?"},
        {"LOWER": {"IN": ["system","technique","performance", "service"]}}
    ],
    [   {"LEMMA": "generate"},
        {"OP": "?"},
        {"OP": "?"},
        {"LOWER": {"IN": ["output","response","content", "information"]}}
    ],
    [   {"LEMMA": "computation"},
        {"OP": "?"},
        {"OP": "?"},
        {"LOWER": {"IN": ["analysis"]}}
    ],
]
},
}


<h2>Step 4: Running PhraseMatcher</h2>

In [ ]:
# load NLP matchers
phrase_matcher = PhraseMatcher(nlp.vocab)
token_matcher = Matcher(nlp.vocab)

In [ ]:
for category, rules in red_flag_dict.items(): # establishes the rubric of your dictionary
    phrase_matcher.add(category, [nlp.make_doc(p) for p in rules.get("phrases", [])])
    token_matcher.add(category, rules.get("patterns", []))

In [ ]:
print("Loaded categories:", list(red_flag_dict.keys())) # verify that the dictionary is properly loaded

In [ ]:
# establish a function to add red flag ‘hits’ into an empty list, covering the entire length of your agreement(s)

def extract_pdf_text(pdf_path):
parts = []
with pdfplumber.open(str(pdf_path)) as pdf:
    for page in pdf.pages:
        parts.append(page.extract_text() or "")
return "\n".join(parts)

pdf_path = pdfs[0]
text = extract_pdf_text(pdf_path)
doc = nlp(text)

matches = []

for match_id, start, end in phrase_matcher(doc):
    span = doc[start:end]
    matches.append((nlp.vocab.strings[match_id], span.text, span.sent.text, "phrase"))

for match_id, start, end in token_matcher(doc):
    span = doc[start:end]
    matches.append((nlp.vocab.strings[match_id], span.text, span.sent.text, "pattern"))

In [ ]:
# sense check, checking the first ten recorded matches (hits)

print("Testing:", pdf_path)
print("Matches:", len(matches))
matches[:10]

# silence pdfplumber warnings, as we have verified that the text has been extracted with enough accuracy (see discussion in main lesson)
warnings.filterwarnings("ignore", message="Could not get FontBBox from font descriptor.*")

<h2>Step 5: Red Flag Exportation</h2>

In [ ]:
# retains .xlsv file structure, with folder headings

rows = []
    for pdf_path in pdfs:
    folder = pdf_path.relative_to(ROOT).parts[0]
    text = extract_pdf_text(pdf_path)
    doc = nlp(text)

for match_id, start, end in phrase_matcher(doc):
    span = doc[start:end]
    rows.append({
        "folder": folder,
        "file": str(pdf_path.relative_to(ROOT)),
        "rule": nlp.vocab.strings[match_id],
        "match_text": span.text,
        "sentence": span.sent.text,
        "method": "phrase",
    })

for match_id, start, end in token_matcher(doc):
    span = doc[start:end]
    rows.append({
        "folder": folder,
        "file": str(pdf_path.relative_to(ROOT)),
        "rule": nlp.vocab.strings[match_id],
        "match_text": span.text,
        "sentence": span.sent.text,
        "method": "pattern",
    })

# silence the spaCy text extraction warning, as we have verified its accuracy

warnings.filterwarnings("ignore", message="Could not get FontBBox from font descriptor.*")

In [ ]:
# drop any remaining duplicates, for fair and robust analysis

df = df.drop_duplicates(subset=["file","rule","sentence","method"])

In [ ]:
# quick view of the first five match (hit) results

df = pd.DataFrame(matches)

df.head()

In [ ]:
# optional, but can also use SpaCy for noun detection and extraction, to direct further assessment of AI risks (if you have a particular use case)

noun_phrases = set()

for chunk in doc.noun_chunks:
    noun_phrases.add(chunk.text.lower())

for np in list(noun_phrases)[:20]:
    print(np) # returns first twenty results


In [ ]:
# for exporting one licence

 out_path = Path("red_flags_by_folder.xlsx")
 with pd.ExcelWriter(out_path, engine="openpyxl")
 out_path

In [ ]:
# if exporting multiple licences

 out_path = Path("red_flags_by_folder.xlsx")
 with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
 for folder in sorted({p.relative_to(ROOT).parts[0] for p in pdfs}):
      sheet = folder[:31]
      sub = df[df["folder"] == folder].drop(columns=["folder"])
      sub.to_excel(writer, sheet_name=sheet, index=False)

 out_path